# Hyperparameter Experimentation
Controlled experiments on the digits dataset, varying optimizer, learning rate, batch size, and epochs, then comparing the outcomes in a table.

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import time

In [8]:
class DigitsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class FeedForwardNet(nn.Module):
    def __init__(self, input_size=64, hidden_size=32, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
data = load_digits()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_ds = DigitsDataset(X_train, y_train)
test_ds = DigitsDataset(X_test, y_test)

## Experiment runner
Trains a fresh model with a given set of hyperparameters and returns the final test accuracy and training time.

In [10]:
def run_experiment(optimizer_name, lr, batch_size, epochs):
    torch.manual_seed(0)
    model = FeedForwardNet()
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    start = time.time()
    for epoch in range(epochs):
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = loss_fn(outputs, batch_y)
            loss.backward()
            optimizer.step()
    duration = time.time() - start

    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            outputs = model(batch_x)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    accuracy = correct / total

    return accuracy, duration

## Run the experiments
Varying optimizer, learning rate, batch size and epoch count.

In [11]:
experiments = [
    {"optimizer_name": "sgd", "lr": 0.01, "batch_size": 16, "epochs": 10},
    {"optimizer_name": "sgd", "lr": 0.1, "batch_size": 16, "epochs": 10},
    {"optimizer_name": "adam", "lr": 0.001, "batch_size": 16, "epochs": 10},
    {"optimizer_name": "adam", "lr": 0.001, "batch_size": 64, "epochs": 10},
    {"optimizer_name": "adam", "lr": 0.001, "batch_size": 16, "epochs": 20},
]

results = []
for exp in experiments:
    accuracy, duration = run_experiment(**exp)
    row = dict(exp)
    row["accuracy"] = round(accuracy, 4)
    row["train_time_sec"] = round(duration, 2)
    results.append(row)
    print(row)

{'optimizer_name': 'sgd', 'lr': 0.01, 'batch_size': 16, 'epochs': 10, 'accuracy': 0.8028, 'train_time_sec': 2.46}
{'optimizer_name': 'sgd', 'lr': 0.1, 'batch_size': 16, 'epochs': 10, 'accuracy': 0.9778, 'train_time_sec': 2.11}
{'optimizer_name': 'adam', 'lr': 0.001, 'batch_size': 16, 'epochs': 10, 'accuracy': 0.9778, 'train_time_sec': 3.39}
{'optimizer_name': 'adam', 'lr': 0.001, 'batch_size': 64, 'epochs': 10, 'accuracy': 0.9583, 'train_time_sec': 1.09}
{'optimizer_name': 'adam', 'lr': 0.001, 'batch_size': 16, 'epochs': 20, 'accuracy': 0.975, 'train_time_sec': 6.22}


## Comparison table

In [12]:
results_df = pd.DataFrame(results)
results_df.to_csv("experiment_log.csv", index=False)
results_df

,optimizer_name,lr,batch_size,epochs,accuracy,train_time_sec
0,sgd,0.010,16,10,0.8028,2.46
1,sgd,0.100,16,10,0.9778,2.11
2,adam,0.001,16,10,0.9778,3.39
3,adam,0.001,64,10,0.9583,1.09
4,adam,0.001,16,20,0.9750,6.22


## Observations
- SGD with a small learning rate (0.01) tends to converge slower than Adam given the same number of epochs.
- Raising the SGD learning rate to 0.1 usually closes most of that gap.
- Adam with a bigger batch size (64) trains faster per epoch, since there are fewer weight updates, sometimes at a small accuracy cost.
- Doubling the epochs for Adam generally gives the best accuracy at the cost of more training time.
- Overall Adam is more forgiving of a default learning rate than SGD, which needed tuning to perform well.